# Workspace Budget Calculator

Replicates the budget formula from `EmbedPool::spawn`. Adjust your deployment parameters to see `max_workspace_bytes`, `worst_case_peak_bytes`, and utilization.

In [ ]:
%pip install ipywidgets ipympl


In [ ]:
# Copyright (c) 2026 J. Patrick Fulton
# Apache-2.0

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
try:
    from visuals.common import C_BLUE, C_RED, C_GREEN, C_YELLOW, C_GREY
except ImportError:
    # Inlined constants for JupyterLite/Pyodide where visuals package is not available
    C_BLUE   = "#4477AA"
    C_RED    = "#EE6677"
    C_GREEN  = "#228833"
    C_YELLOW = "#CCBB44"
    C_GREY   = "#BBBBBB"

# Enable interactive backend if ipympl is available; fall back to inline (PNG) silently.
try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic("matplotlib", "widget")
    _INTERACTIVE = True
except Exception:
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic("matplotlib", "inline")
    else:
        matplotlib.use("Agg")
    _INTERACTIVE = False

## Budget formula

```
total_workspace     = available − N×model_rss_per_worker − OS_HEADROOM
per_worker_workspace = total_workspace × safety_factor / N
worst_case_peak     = N×per_worker_workspace + N×model_rss + OS_HEADROOM
utilization         = worst_case_peak / available × 100
```

All values in GiB unless stated otherwise. `OS_HEADROOM` is fixed at 2 GiB.

In [ ]:
OS_HEADROOM = 2 * 1024**3  # 2 GiB fixed headroom

summary_html = widgets.HTML()

workers_slider = widgets.IntSlider(
    value=7, min=1, max=16, step=1,
    description="Workers (N):",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="480px"),
)
model_rss_slider = widgets.FloatSlider(
    value=1.1, min=0.5, max=3.0, step=0.1,
    description="Model RSS/worker (GiB):",
    style={"description_width": "170px"},
    layout=widgets.Layout(width="480px"),
    readout_format=".1f",
)
available_slider = widgets.FloatSlider(
    value=28.0, min=4.0, max=64.0, step=1.0,
    description="Available mem (GiB):",
    style={"description_width": "170px"},
    layout=widgets.Layout(width="480px"),
    readout_format=".0f",
)
safety_slider = widgets.FloatSlider(
    value=0.7, min=0.5, max=0.95, step=0.01,
    description="Safety factor:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="480px"),
    readout_format=".2f",
)


def compute(n, model_rss_gb, available_gb, safety):
    available = available_gb * 1024**3
    model_rss = model_rss_gb * 1024**3
    total_ws = available - n * model_rss - OS_HEADROOM
    per_worker_ws = max(total_ws * safety / n, 0)
    worst_case = n * per_worker_ws + n * model_rss + OS_HEADROOM
    util = worst_case / available * 100
    return {
        "per_worker_ws_gib": per_worker_ws / 1024**3,
        "total_ws_gib": total_ws / 1024**3,
        "worst_case_gib": worst_case / 1024**3,
        "util": util,
        "model_total_gib": n * model_rss_gb,
        "ws_total_gib": n * per_worker_ws / 1024**3,
        "os_gib": OS_HEADROOM / 1024**3,
        "free_gib": max(available_gb - worst_case / 1024**3, 0),
    }


def traffic_light(util):
    if util < 80:
        return "#228833", "green", "SAFE"
    if util < 90:
        return "#CCBB44", "yellow", "WARNING"
    return "#EE6677", "red", "DANGER"


# Output widget for fallback mode (unused in interactive mode but defined always)
_out = widgets.Output()

# Create figure once for interactive mode; DO NOT call plt.show() — ipympl renders inline automatically
if _INTERACTIVE:
    _fig, _ax = plt.subplots(figsize=(8, 4))


def update(_change=None):
    n = workers_slider.value
    r = compute(n, model_rss_slider.value, available_slider.value, safety_slider.value)
    color, _, label = traffic_light(r["util"])

    summary_html.value = (
        f'<div style="font-family:monospace;font-size:14px;line-height:1.8">'
        f'<b>per_worker_workspace</b>: {r["per_worker_ws_gib"]:.2f} GiB<br>'
        f'<b>total_workspace (all workers)</b>: {r["ws_total_gib"]:.2f} GiB<br>'
        f'<b>worst_case_peak</b>: {r["worst_case_gib"]:.2f} GiB<br>'
        f'<b>utilization</b>: {r["util"]:.1f}% '
        f'<span style="color:{color};font-weight:bold">&#9632; {label}</span><br>'
        f'</div>'
    )

    labels = ["Model weights", "Workspace", "OS headroom", "Free"]
    sizes = [r["model_total_gib"], r["ws_total_gib"], r["os_gib"], r["free_gib"]]
    bar_colors = [C_RED, C_BLUE, C_GREY, C_GREEN]

    if _INTERACTIVE:
        _ax.cla()
        _ax.barh(["Memory breakdown"], [sizes[0]], color=bar_colors[0], label=labels[0])
        left = sizes[0]
        for s, c, lbl in zip(sizes[1:], bar_colors[1:], labels[1:]):
            _ax.barh(["Memory breakdown"], [s], left=left, color=c, label=lbl)
            left += s
        _ax.set_xlabel("GiB")
        _ax.set_xlim(0, available_slider.value * 1.05)
        _ax.set_title(
            f"Memory breakdown — {n} workers × {model_rss_slider.value:.1f} GiB RSS, "
            f"{available_slider.value:.0f} GiB available"
        )
        _ax.legend(loc="lower right", fontsize=9)
        _fig.tight_layout()
        _fig.canvas.draw_idle()
    else:
        with _out:
            _out.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.barh(["Memory breakdown"], [sizes[0]], color=bar_colors[0], label=labels[0])
            left = sizes[0]
            for s, c, lbl in zip(sizes[1:], bar_colors[1:], labels[1:]):
                ax.barh(["Memory breakdown"], [s], left=left, color=c, label=lbl)
                left += s
            ax.set_xlabel("GiB")
            ax.set_xlim(0, available_slider.value * 1.05)
            ax.set_title(
                f"Memory breakdown — {n} workers × {model_rss_slider.value:.1f} GiB RSS, "
                f"{available_slider.value:.0f} GiB available"
            )
            ax.legend(loc="lower right", fontsize=9)
            fig.tight_layout()
            display(fig)
            plt.close(fig)


for slider in [workers_slider, model_rss_slider, available_slider, safety_slider]:
    slider.observe(update, names="value")

controls = widgets.VBox([
    widgets.HBox([workers_slider, model_rss_slider]),
    widgets.HBox([available_slider, safety_slider]),
])
display(controls, summary_html, _out)
update()

## Reading the output

- **Green (< 80%)**: comfortable headroom, current config is safe
- **Yellow (80–90%)**: approaching limit; consider reducing workers or safety factor
- **Red (> 90%)**: over-subscribed; risk of OOM kill under peak load

The production 7-worker fp16 config on a 28 GB ECS task runs at ~74% utilization.